<style>
.custom-spacing {
    line-height: 0.5;
}
</style>

### <center>**TUGAS INDIVIDU**</center>
### <center>**KECERDASAN ARTIFISIAL UNTUK GEOFISIKA**</center>
#### <center>**Dosen Pengampu:** 
#### <center>**Dr. Muhammad Alimuddin Hamzah, M. Eng**</center>
#### <center>**Prof.  Dr. Samsu Arif, M. Si**</center>
#### <center>**Muhammad Farid Wajedy, S.Si, M.Si**</center>

<div style="text-align:center;">
  <img src="https://www.unhas.ac.id/wp-content/uploads/2022/09/Logo-Resmi-Unhas-1.png" alt="Deskripsi Gambar" style="width:280px;">
</div>

### <center>**DISUSUN OLEH :**</center>

#### <center>**SITI SARAH CHAIRANI           (H061241022)**<center>

### <center>**PROGRAM STUDI GEOFISIKA**</center>
### <center>**FAKULTAS MATEMATIKA DAN ILMU PENGETAHUAN ALAM**</center>
#### <center>**UNIVERSITAS HASANUDDIN**</center>
#### <center>**MAKASSAR**</center>
#### <center>**2026**</center>

<hr>

# Form Prompt AI untuk Geofisika (Multi-Metode)

Struktur mengikuti Bab 2 *AI Cookbook untuk Earth Sciences* (prompt sebagai *work specification*), sesuai diagram pada Gambar 2.1.

<div align="center">

| No | Metode Geofisika | Topik Prompt |
|----|------------------|--------------|
| 1 | Geolistrik (VES) | Perhitungan resistivitas semu dan fitting model 1D tiga lapisan |
| 2 | Seismik refraksi | Estimasi kecepatan lapisan dan kedalaman bidang bias (metode intercept-time) |
| 3 | Gravitasi | Filter derivatif (FHD dan SVD) untuk delineasi struktur dari grid anomali Bouguer |
| 4 | Seismologi | Estimasi magnitude of completeness (Mc) dan b-value Gutenberg-Richter |
| 5 | Magnetotellurik (MT) | QC kurva sounding MT dan transformasi Niblett-Bostick |

</div>

<br>

<div align="center">

| Kode | Komponen | Fungsi |
|------|----------|--------|
| **R** | Role | Sudut pandang kerja, bukan credential |
| **T** | Task | Verb yang bisa diperiksa + objek + domain + kondisi penerimaan |
| **C** | Context | Tujuan, audiens, workflow, keputusan yang didukung |
| **I** | Input | Path, format, variabel, unit, CRS/datum, provenance |
| **K** | Constraints | File yang tidak boleh diubah, method, package, jaringan, approval |
| **O** | Output | Kontrak output: path, kolom/unit, format (rencana teks, belum hasil) |
| **Q** | Quality criteria | Kriteria yang dapat dinilai eksplisit |
| **V** | Optional verification | Check + expected + tindakan bila gagal |

</div>

> Bagian **O** adalah *expected output* berupa narasi rencana, bukan *actual result* dan bukan gambar.

## Template Umum

```markdown
## [R] Role
Bertindaklah sebagai <sudut pandang kerja>. Fokus pada <apa yang diperiksa>.

## [T] Task
<Verb> <objek> <domain/kondisi>, lalu <verb kedua>. <Kondisi penerimaan>.

## [C] Context
<Tujuan>, <audiens>, <tahap workflow>. Hasil dipakai untuk <keputusan>, bukan untuk <hal yang tidak dilakukan>.

## [I] Input
- `<path>`: format, variabel/kolom, unit, CRS/datum, sumber

## [K] Constraints
- Jangan mengubah <file/folder>.
- Method: <yang boleh>; <yang dilarang>.
- Software: <yang tersedia>; tanpa jaringan.
- Bila <kondisi tak jelas>: berhenti dan tanyakan.

## [O] Output (rencana teks)
- `<path output>`: <isi, kolom, unit>

## [Q] Quality criteria
- <kriteria terukur>

## [V] Optional verification
- Check: <command>. Expected: <kondisi>. Jika gagal: <tindakan>.
```

## Prompt 1 — Geolistrik: Resistivitas Semu dan Fitting Model 1D pada Data VES

### [R] Role
Bertindaklah sebagai reviewer interpretasi geolistrik 1D. Fokus pada ketepatan perhitungan resistivitas semu, kualitas kurva sounding, dan ketidakunikan (ekuivalensi) model lapisan.

### [T] Task
Hitung resistivitas semu (ρa) dari data Vertical Electrical Sounding konfigurasi Schlumberger `data/raw/ves_01.csv`, tandai titik yang menyimpang pada kurva ρa terhadap AB/2, lalu lakukan fitting model 1D dengan 3 lapisan dan laporkan resistivitas, ketebalan, serta misfit-nya. Jangan menafsirkan litologi atau potensi akuifer.

### [C] Context
Survei VES pada satu titik di dataran aluvial untuk investigasi awal air tanah. Hasil dipakai tim hidrogeologi untuk memutuskan titik mana yang perlu diverifikasi lebih lanjut. Hasil bukan penetapan lokasi atau kedalaman sumur bor.

### [I] Input
- `data/raw/ves_01.csv`: kolom `ab2_m` (m), `mn2_m` (m), `I_mA` (mA), `dV_mV` (mV). Bentangan AB/2 maksimum `{{m}}`. Sumber alat: `{{merek/model}}`.
- Faktor geometri Schlumberger: `K = π · ((AB/2)² − (MN/2)²) / MN`, dengan `MN = 2 · (MN/2)`. Rumus ini yang dipakai dan harus dicatat.
- Jumlah lapisan model: `3`. Rentang awal resistivitas: `{{min–max Ω·m}}`.

### [K] Constraints
- Jangan mengubah `data/raw/`; jangan menghapus baris data.
- Titik outlier hanya diberi flag dengan ambang `{{ambang}}` yang dinyatakan; jangan dibuang atau dihaluskan diam-diam.
- Bila terjadi lompatan ρa saat MN diganti (segmen kurva), laporkan; jangan menggeser (*shifting*) segmen tanpa persetujuan.
- Model tetap 3 lapisan; jangan menambah atau mengurangi lapisan sendiri.
- Software: Python dengan NumPy, SciPy, dan pandas; tanpa jaringan dan tanpa software inversi tertutup.
- Jika `mn2_m` lebih besar dari `ab2_m` atau satuan tidak jelas: berhenti dan tanyakan.

### [O] Output (rencana teks)
- `results/ves/ves_01_rhoa.csv`: `ab2_m`, `mn2_m`, `K_m`, `rho_a_ohm_m`, `flag`, `note`.
- `results/ves/ves_01_model.csv`: `layer`, `rho_ohm_m`, `thickness_m`, `depth_top_m`, `depth_bottom_m` (lapisan ke-3 dengan ketebalan tak hingga).
- `results/ves/ves_01_report.md`: narasi jumlah titik data, titik yang di-flag, hasil fitting, nilai RMS misfit (%), dan catatan ekuivalensi model.
- `results/ves/run_config.json`: rumus K, ambang flag, tebakan awal, algoritma optimasi, versi package.

### [Q] Quality criteria
- Jumlah baris `ves_01_rhoa.csv` sama dengan jumlah baris input.
- Setiap flag memuat nilai, ambang, dan alasan.
- Satuan (m, Ω·m) eksplisit pada semua kolom.
- Laporan memisahkan fakta terukur, asumsi model, dan hal yang belum bisa ditafsirkan.
- Nilai misfit dilaporkan bersama pernyataan bahwa lebih dari satu model dapat cocok dengan data.

### [V] Optional verification
- Check: buat data sintetis dari model 3 lapisan yang diketahui, lalu jalankan prosedur yang sama. Expected: parameter model terpulihkan dalam `{{toleransi}}`%. Jika gagal: laporkan bug pada forward model atau optimasi.
- Check: hitung ulang `rho_a` untuk 5 baris acak dengan script terpisah. Expected: sama dengan `ves_01_rhoa.csv`. Jika gagal: periksa rumus K dan satuan MN.
- Check: SHA-256 `ves_01.csv` sebelum dan sesudah. Expected: identik. Jika gagal: hentikan dan laporkan.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

folder = Path("data/raw")
folder.mkdir(parents=True, exist_ok=True)

ab2 = np.array([1, 1.5, 2, 3, 4, 6, 8, 10, 15, 20, 30, 40, 60, 80, 100])
mn2 = np.where(ab2 <= 10, 0.5, 2.5)
rho_a = np.array([40, 38, 36, 32, 28, 22, 20, 22, 30, 40, 60, 75, 90, 95, 98])  # Ohm.m (contoh)
I_mA = np.full(len(ab2), 100.0)

K = np.pi * (ab2**2 - mn2**2) / (2 * mn2)
dV_mV = rho_a * I_mA / K

pd.DataFrame({"ab2_m": ab2, "mn2_m": mn2, "I_mA": I_mA, "dV_mV": dV_mV}).to_csv(folder / "ves_01.csv", index=False)
print("File dibuat di:", (folder / "ves_01.csv").resolve())

File dibuat di: D:\Semester 5\Metode AI\data\raw\ves_01.csv


In [3]:
from pathlib import Path
import json
import hashlib
import platform
import sys

import numpy as np
import pandas as pd

METHOD = "VES"  # pilih: VES / REFRACTION / GRAVITY / SEISMICITY / MT
BASE_DIR = Path(".")
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if METHOD.upper() == "VES":
    INPUT_FILE = BASE_DIR / "data" / "raw" / "ves_01.csv"
    OUT_DIR = RESULTS_DIR / "ves"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Sesuaikan parameter ini dengan prompt/tugas
    RHOA_FLAG_THRESHOLD = 3.0       # flag bila beda log10 rhoa berurutan > nilai ini
    RHO_MIN = 1.0                  # ohm m; batas awal model (sesuaikan)
    RHO_MAX = 1000.0               # ohm m; batas awal model (sesuaikan)
    N_LAYERS = 3
    INSTRUMENT = "belum diisi"

    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"File input tidak ditemukan: {INPUT_FILE.resolve()}")

    SHA_BEFORE = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
    df = pd.read_csv(INPUT_FILE)
    required = ["ab2_m", "mn2_m", "I_mA", "dV_mV"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Kolom input VES tidak ditemukan: {missing}")

    # Validasi tanpa menghapus baris
    if df[required].isna().any().any():
        raise ValueError("Ada nilai kosong pada kolom wajib VES. Periksa data; baris tidak dihapus.")
    if (df["ab2_m"] <= 0).any() or (df["mn2_m"] <= 0).any():
        raise ValueError("AB/2 dan MN/2 harus positif.")
    if (df["mn2_m"] >= df["ab2_m"]).any():
        raise ValueError("MN/2 harus lebih kecil dari AB/2. Proses dihentikan.")
    if (df["I_mA"] == 0).any():
        raise ValueError("Arus I_mA bernilai nol; rhoa tidak dapat dihitung.")

    # K = pi * ((AB/2)^2 - (MN/2)^2) / MN, dengan MN = 2*(MN/2)
    mn_m = 2.0 * df["mn2_m"]
    K_m = np.pi * (df["ab2_m"]**2 - df["mn2_m"]**2) / mn_m
    # dV/I: mV/mA = V/A, sehingga hasil rhoa dalam ohm m
    rhoa = K_m * (df["dV_mV"] / df["I_mA"])

    out = df[["ab2_m", "mn2_m"]].copy()
    out["K_m"] = K_m
    out["rho_a_ohm_m"] = rhoa
    out["flag"] = ""
    out["note"] = ""

    # Flag lonjakan berdasarkan selisih log10 rhoa antar titik berurutan
    valid_rhoa = rhoa > 0
    log_rhoa = pd.Series(np.nan, index=df.index)
    log_rhoa.loc[valid_rhoa] = np.log10(rhoa.loc[valid_rhoa])
    jump = log_rhoa.diff().abs()
    out.loc[jump > RHOA_FLAG_THRESHOLD, "flag"] = "CHECK_JUMP"
    out.loc[jump > RHOA_FLAG_THRESHOLD, "note"] = (
        "Perubahan log10 rhoa berurutan melebihi ambang "
        + str(RHOA_FLAG_THRESHOLD) + "; periksa segmen MN dan data lapangan."
    )
    out.loc[~valid_rhoa, "flag"] = "INVALID_RHOA"
    out.loc[~valid_rhoa, "note"] = "rho_a <= 0; periksa tanda beda potensial dan polaritas."

    out.to_csv(OUT_DIR / "ves_01_rhoa.csv", index=False)

    # Model CSV dibuat sebagai template, bukan hasil inversi palsu.
    model_template = pd.DataFrame({
        "layer": [1, 2, 3],
        "rho_ohm_m": [np.nan, np.nan, np.nan],
        "thickness_m": [np.nan, np.nan, np.nan],
        "depth_top_m": [0.0, np.nan, np.nan],
        "depth_bottom_m": [np.nan, np.nan, np.inf]
    })
    model_template.to_csv(OUT_DIR / "ves_01_model.csv", index=False)

    # Catat integritas input dan konfigurasi
    SHA_AFTER = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
    config = {
        "method": "VES Schlumberger",
        "input": str(INPUT_FILE),
        "geometry_factor": "K = pi*((AB/2)^2-(MN/2)^2)/MN; MN=2*(MN/2)",
        "rho_a_formula": "rho_a = K * dV/I; mV/mA = V/A",
        "outlier_flag_threshold_abs_delta_log10_rhoa": RHOA_FLAG_THRESHOLD,
        "number_of_layers_requested": N_LAYERS,
        "initial_resistivity_range_ohm_m": [RHO_MIN, RHO_MAX],
        "instrument": INSTRUMENT,
        "python_version": sys.version,
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "sha256_before": SHA_BEFORE,
        "sha256_after": SHA_AFTER,
        "input_unchanged": SHA_BEFORE == SHA_AFTER,
        "note": "Model CSV masih template. Fitting 1D tidak dijalankan karena perlu forward solver Schlumberger tervalidasi."
    }
    (OUT_DIR / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

    report = f"""# Laporan VES

- Jumlah baris input: {len(df)}
- Jumlah rhoa positif: {int(valid_rhoa.sum())}
- Jumlah baris berflag: {int((out['flag'] != '').sum())}
- Faktor geometri: K = pi*((AB/2)^2-(MN/2)^2)/MN, dengan MN=2*(MN/2).
- Ambang flag: |delta log10(rhoa)| > {RHOA_FLAG_THRESHOLD}.
- SHA-256 input sebelum/sesudah sama: {SHA_BEFORE == SHA_AFTER}.
- Resistivitas semu dihitung dan disimpan pada `ves_01_rhoa.csv`.
- Fitting model 1D tiga lapisan belum dilakukan. `ves_01_model.csv` adalah template kosong, bukan hasil inversi.
- Diperlukan forward solver Schlumberger 1D yang tervalidasi sebelum parameter resistivitas, ketebalan, dan RMS misfit dapat dilaporkan.
- Hasil resistivitas semu maupun model tidak langsung menentukan litologi atau potensi akuifer.
"""
    (OUT_DIR / "ves_01_report.md").write_text(report, encoding="utf-8")
    print("VES selesai: rhoa dan QC disimpan di", OUT_DIR.resolve())

VES selesai: rhoa dan QC disimpan di D:\Semester 5\Metode AI\results\ves


---

## Prompt 2 — Seismik Refraksi: Kecepatan Lapisan dan Kedalaman Bidang Bias (Intercept-Time)

### [R] Role
Bertindaklah sebagai reviewer interpretasi seismik refraksi dangkal. Fokus pada pemisahan segmen gelombang langsung dan gelombang bias, konsistensi kecepatan forward–reverse, dan asumsi geometri lapisan.

### [T] Task
Dari picks first-break `data/raw/refraction_picks.csv` (shot forward dan reverse), tentukan segmen gelombang langsung dan gelombang bias dengan regresi linear terpisah, hitung V1, V2, intercept time, dan crossover distance, lalu hitung kedalaman bidang bias h1 dengan metode intercept-time untuk kedua arah shot. Jangan menjalankan tomografi atau inversi.

### [C] Context
Survei refraksi dangkal untuk memperkirakan kedalaman batuan dasar pada lokasi rencana bangunan. Hasil dipakai sebagai estimasi awal oleh geoteknik. Hasil tidak dipakai untuk menetapkan kapasitas dukung tanah atau desain pondasi.

### [I] Input
- `data/raw/refraction_picks.csv`: `shot_id`, `shot_x_m` (m), `receiver_x_m` (m), `t_fb_ms` (ms), `direction` (`forward`/`reverse`).
- Spasi geophone: `{{m}}`; jumlah geophone: `{{n}}`.
- Model: 2 lapisan, bidang bias dianggap datar. Rumus: `h1 = (t_i · V1 · V2) / (2 · √(V2² − V1²))`, dengan `t_i` intercept time.
- Titik pemisah segmen (breakpoint): `{{manual: offset m / otomatis: kriteria}}`.

### [K] Constraints
- Jangan mengubah file picks; jangan melakukan picking ulang dari trace.
- Breakpoint harus dinyatakan (manual atau kriteria otomatis) dan dicatat; jangan dipilih diam-diam.
- Model tetap 2 lapisan; jangan menambah lapisan ketiga.
- Jika V2 forward dan V2 reverse berbeda lebih dari `{{ambang}}`%, beri flag "indikasi bidang miring atau heterogen" dan jangan merata-ratakan tanpa keterangan.
- Software: Python, NumPy, pandas; tanpa jaringan.
- Jika V2 ≤ V1: berhenti dan laporkan (rumus tidak valid).

### [O] Output (rencana teks)
- `results/refraction/velocity_fit.csv`: `direction`, `V1_m_s`, `V2_m_s`, `intercept_ms`, `crossover_m`, `r2_direct`, `r2_refracted`, `n_direct`, `n_refracted`.
- `results/refraction/depth_estimate.csv`: `direction`, `h1_m`, `V_ratio_V2_V1`, `flag`.
- `results/refraction/report.md`: narasi pemilihan breakpoint, hasil V1, V2, dan h1 tiap arah, perbedaan forward–reverse, serta keterbatasan asumsi bidang datar.
- `results/refraction/run_config.json`: rumus, breakpoint, versi package.

### [Q] Quality criteria
- Setiap kecepatan dilaporkan bersama R² dan jumlah titik yang dipakai.
- Satuan m/s, ms, dan m eksplisit pada semua kolom.
- Perbedaan h1 forward–reverse dilaporkan, tidak disembunyikan.
- Picks yang tidak dipakai pada regresi tetap tercatat beserta alasannya.

### [V] Optional verification
- Check: buat picks sintetis dari model 2 lapisan (V1 = 400 m/s, V2 = 1800 m/s, h1 = 5 m) dan jalankan prosedur. Expected: V1, V2, h1 terpulihkan dalam `{{toleransi}}`%. Jika gagal: laporkan bug.
- Check: bandingkan crossover distance terhitung dengan pendekatan `x_c = 2 · h1 · √((V2 + V1)/(V2 − V1))`. Expected: selisih kecil. Jika besar: periksa breakpoint.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

In [ ]:
import os, sys, json, hashlib, platform
import numpy as np
import pandas as pd
import scipy
from scipy.special import j1

# ----------------------------- PARAMETER (isi sesuai prompt) -----------------
LOKASI        = "{{lokasi}}"
LITOLOGI      = "{{aluvial/batupasir/vulkanik}}"
N_LAYERS      = 3                  # jumlah lapisan awal {{3/4}}
RHO_MIN       = 1.0                # batas resistivitas (ohm.m)
RHO_MAX       = 1000.0
H_MIN         = 0.3                # batas ketebalan lapisan (m)
JUMP_LOG10    = 0.5                # ambang lonjakan antar titik AB/2 (selisih log10 rho_a)
AQ_RHO        = (30.0, 200.0)      # rentang resistivitas kandidat akuifer (ohm.m) - asumsi
MAX_ITER      = 60
VERIFIKASI_JML_LAPISAN = True      # ulangi inversi 3 & 4 lapisan pada 2 titik
BUAT_DATA_SINTETIS = True          # True: bila ves_points.csv belum ada, dibuat data SIMULASI otomatis

PATH_VES      = "data/raw/ves_points.csv"
PATH_BORELOG  = "data/raw/borelog.csv"
OUT_DIR       = "results/ves"

# ----------------------------- DATA SINTETIS (bila file belum ada) -----------
# PERHATIAN: ini data simulasi untuk uji skrip, BUKAN data lapangan.
if (not os.path.exists(PATH_VES)) and BUAT_DATA_SINTETIS:
    os.makedirs(os.path.dirname(PATH_VES), exist_ok=True)
    lam_s = np.logspace(-4, 2.5, 12000)
    dl_s = np.log(lam_s[1] / lam_s[0])
    model_s = {"VES1": ([40, 8, 120], [3, 12]),
               "VES2": ([60, 150, 15], [2, 20]),
               "VES3": ([30, 90, 10], [4, 15])}
    AB2_s = np.array([1, 1.5, 2, 3, 4, 6, 8, 10, 15, 20, 30, 40, 60, 80, 100.0])
    rng_s = np.random.default_rng(1)
    baris_s = []
    for pid_s in model_s:
        rho_s, h_s = model_s[pid_s]
        MN2_s = np.where(AB2_s < 10, 0.5, 5.0)
        T_s = np.full_like(lam_s, rho_s[-1])
        for i_s in range(1, -1, -1):
            th_s = np.tanh(lam_s * h_s[i_s])
            T_s = (T_s + rho_s[i_s] * th_s) / (1 + T_s * th_s / rho_s[i_s])
        ra_s = rho_s[0] + AB2_s**2 * ((T_s - rho_s[0])[None, :] * j1(lam_s[None, :] * AB2_s[:, None]) * lam_s**2 * dl_s).sum(axis=1)
        ra_s = ra_s * np.exp(rng_s.normal(0, 0.02, len(AB2_s)))
        K_s = np.pi * (AB2_s**2 - MN2_s**2) / (2 * MN2_s)
        for a_s in range(len(AB2_s)):
            baris_s.append([pid_s, AB2_s[a_s], MN2_s[a_s], 100.0, ra_s[a_s] / K_s[a_s] * 100.0, -5.1, 119.5])
    pd.DataFrame(baris_s, columns=["point_id", "AB2_m", "MN2_m", "I_mA", "dV_mV", "lat", "lon"]).to_csv(PATH_VES, index=False)
    print("PERINGATAN: %s tidak ditemukan -> dibuat DATA SINTETIS di %s" % (PATH_VES, os.path.abspath(PATH_VES)))

if not os.path.exists(PATH_VES):
    sys.exit("File tidak ditemukan: %s\nFolder kerja saat ini: %s\nLetakkan ves_points.csv di sana atau set BUAT_DATA_SINTETIS = True." % (PATH_VES, os.getcwd()))

# ----------------------------- CEK SHA-256 SEBELUM ---------------------------
f = open(PATH_VES, "rb"); sha_sebelum = hashlib.sha256(f.read()).hexdigest(); f.close()
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------- BACA DATA -------------------------------------
df = pd.read_csv(PATH_VES)
kolom_wajib = ["point_id", "AB2_m", "MN2_m", "I_mA", "dV_mV", "lat", "lon"]
for k in kolom_wajib:
    if k not in df.columns:
        sys.exit("Kolom wajib tidak ada: " + k)
df["point_id"] = df["point_id"].astype(str)

# ----------------------------- RESISTIVITAS SEMU -----------------------------
# K = pi (L^2 - l^2) / (2 l), L = AB/2, l = MN/2 ; rho_a = K * dV / I
L_ = df["AB2_m"].astype(float)
l_ = df["MN2_m"].astype(float)
df["K_m"] = np.pi * (L_**2 - l_**2) / (2 * l_)
df["rho_a_ohm_m"] = df["K_m"] * (df["dV_mV"] / df["I_mA"])     # mV/mA = ohm

# ----------------------------- QC FLAG (tidak ada baris dihapus) -------------
flags = [[] for _ in range(len(df))]
for i in range(len(df)):
    r = df.iloc[i]
    nilai = [r["AB2_m"], r["MN2_m"], r["I_mA"], r["dV_mV"]]
    if any(pd.isna(nilai)):
        flags[i].append("data_hilang")
    elif r["MN2_m"] >= r["AB2_m"] or r["AB2_m"] <= 0 or r["MN2_m"] <= 0:
        flags[i].append("geometri_tidak_valid")
    elif r["I_mA"] <= 0:
        flags[i].append("arus_nonpositif")
    elif (not np.isfinite(r["rho_a_ohm_m"])) or r["rho_a_ohm_m"] <= 0:
        flags[i].append("rho_a_nonpositif")
FLAG_FATAL = ["data_hilang", "geometri_tidak_valid", "arus_nonpositif", "rho_a_nonpositif"]

# lonjakan antar titik AB/2 berurutan (hanya flag, data tetap dipakai)
for pid in df["point_id"].unique():
    idx = df.index[(df["point_id"] == pid)].tolist()
    idx = df.loc[idx].sort_values("AB2_m").index.tolist()
    ok = [j for j in idx if not any(x in FLAG_FATAL for x in flags[j])]
    for a in range(1, len(ok)):
        d = abs(np.log10(df.loc[ok[a], "rho_a_ohm_m"]) - np.log10(df.loc[ok[a-1], "rho_a_ohm_m"]))
        if d > JUMP_LOG10:
            flags[ok[a]].append("lonjakan")
df["qc_flag"] = [";".join(x) if len(x) > 0 else "ok" for x in flags]

df_rho = df[["point_id", "AB2_m", "K_m", "rho_a_ohm_m", "qc_flag"]]
df_rho.to_csv(os.path.join(OUT_DIR, "ves_apparent_rho.csv"), index=False)

# ----------------------------- DAFTAR PEKERJAAN INVERSI ----------------------
titik = list(dict.fromkeys(df["point_id"].tolist()))
jobs = [(p, N_LAYERS, "utama") for p in titik]
if VERIFIKASI_JML_LAPISAN:
    for p in titik[:2]:
        for nl in (3, 4):
            jobs.append((p, nl, "verifikasi"))

# ----------------------------- GRID BILANGAN GELOMBANG -----------------------
NLAM = 12000
lam = np.logspace(-4, 2.5, NLAM)            # 1/m
dlnlam = np.log(lam[1] / lam[0])

hasil_lapisan = []     # baris ves_layers.csv (utama)
hasil_verif = []
detail_titik = {}      # untuk interpretasi

for (pid, NL, tag) in jobs:
    d = df[(df["point_id"] == pid)]
    d = d[[not any(x in FLAG_FATAL for x in flags[j]) for j in d.index]]
    d = d.sort_values("AB2_m")
    obs = d["rho_a_ohm_m"].values.astype(float)
    L = d["AB2_m"].values.astype(float)
    npar = 2 * NL - 1

    # jumlah lapisan harus bisa dibenarkan data: n data > n parameter
    if len(obs) < npar + 1:
        if tag == "utama":
            sys.exit("BERHENTI: titik %s hanya punya %d data valid, tidak cukup untuk %d lapisan "
                     "(%d parameter). Kurangi jumlah lapisan atau tambah data. Mohon konfirmasi."
                     % (pid, len(obs), NL, npar))
        else:
            continue

    # matriks kernel Hankel: (nL, nlam)
    J = j1(lam[None, :] * L[:, None]) * (lam ** 2)[None, :] * dlnlam

    # model awal (log10)
    kel = np.array_split(np.log10(obs), NL)
    rho0 = [np.mean(g) for g in kel]
    h0 = np.log10(np.geomspace(max(L.min(), H_MIN * 2), L.max() / 3.0, NL - 1))
    p = np.r_[rho0, h0]
    p = np.clip(p, np.r_[np.full(NL, np.log10(RHO_MIN)), np.full(NL - 1, np.log10(H_MIN))],
                   np.r_[np.full(NL, np.log10(RHO_MAX)), np.full(NL - 1, np.log10(L.max()))])
    lo = np.r_[np.full(NL, np.log10(RHO_MIN)), np.full(NL - 1, np.log10(H_MIN))]
    hi = np.r_[np.full(NL, np.log10(RHO_MAX)), np.full(NL - 1, np.log10(L.max()))]
    model_awal = (10 ** p).tolist()

    mus = np.array([1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1e3])
    eps = 1e-4
    stage = "jac"
    it = 0
    cost = None
    Jm = None; rres = None; trials = None
    selesai = False
    while not selesai:
        # ---- 1 blok forward Pekeris/Hankel (dipakai bergantian untuk Jacobian & trial) ----
        if stage == "jac":
            PB = np.vstack([p, p[None, :] + eps * np.eye(npar)])
        else:
            PB = trials
        rho_b = 10 ** PB[:, :NL]
        h_b = 10 ** PB[:, NL:]
        T = np.repeat(rho_b[:, NL-1:NL], NLAM, axis=1)
        for i in range(NL - 2, -1, -1):
            th = np.tanh(lam[None, :] * h_b[:, i:i+1])
            r_i = rho_b[:, i:i+1]
            T = (T + r_i * th) / (1.0 + T * th / r_i)
        dT = T - rho_b[:, 0:1]
        rho_calc = rho_b[:, 0:1] + (L[None, :] ** 2) * (dT @ J.T)
        rho_calc = np.maximum(rho_calc, 1e-6)
        logF = np.log10(rho_calc)
        # -----------------------------------------------------------------------------------
        if stage == "jac":
            rres = np.log10(obs) - logF[0]
            cost = float(np.sum(rres ** 2))
            Jm = ((logF[1:] - logF[0][None, :]) / eps).T          # (nL, npar)
            A = Jm.T @ Jm
            g = Jm.T @ rres
            trials = []
            for mu in mus:
                step = np.linalg.solve(A + mu * np.diag(np.diag(A) + 1e-9), g)
                trials.append(np.clip(p + step, lo, hi))
            trials = np.array(trials)
            stage = "trial"
        else:
            costs = np.sum((np.log10(obs)[None, :] - logF) ** 2, axis=1)
            k = int(np.argmin(costs))
            it += 1
            if costs[k] < cost:
                rel = (cost - costs[k]) / max(cost, 1e-12)
                p = trials[k].copy()
                if rel < 1e-6 or it >= MAX_ITER:
                    selesai = True
                stage = "jac"
            else:
                selesai = True

    # ---- hitung ulang respons akhir & RMS (ulang forward sekali, p final) ----
    PB = p[None, :]
    rho_b = 10 ** PB[:, :NL]; h_b = 10 ** PB[:, NL:]
    T = np.repeat(rho_b[:, NL-1:NL], NLAM, axis=1)
    for i in range(NL - 2, -1, -1):
        th = np.tanh(lam[None, :] * h_b[:, i:i+1]); r_i = rho_b[:, i:i+1]
        T = (T + r_i * th) / (1.0 + T * th / r_i)
    calc = (rho_b[:, 0:1] + (L[None, :] ** 2) * ((T - rho_b[:, 0:1]) @ J.T))[0]
    rms_pct = float(np.sqrt(np.mean(((obs - calc) / obs) ** 2)) * 100.0)

    # ---- ketidakpastian (ekuivalensi) dari matriks sensitivitas ----
    dof = max(len(obs) - npar, 1)
    s2 = float(np.sum((np.log10(obs) - np.log10(calc)) ** 2)) / dof
    cov = np.linalg.pinv(Jm.T @ Jm) * s2
    sd = np.sqrt(np.abs(np.diag(cov)))
    faktor_rho = 10 ** sd[:NL]

    rho_fin = 10 ** p[:NL]
    h_fin = 10 ** p[NL:]
    top = np.r_[0.0, np.cumsum(h_fin)]
    bot = np.r_[np.cumsum(h_fin), np.nan]
    baris = []
    for i in range(NL):
        baris.append({
            "point_id": pid, "layer_no": i + 1,
            "depth_top_m": round(float(top[i]), 3),
            "depth_bottom_m": (round(float(bot[i]), 3) if i < NL - 1 else np.nan),
            "thickness_m": (round(float(h_fin[i]), 3) if i < NL - 1 else np.nan),
            "resistivity_ohm_m": round(float(rho_fin[i]), 2),
            "rms_pct": round(rms_pct, 2)})
    if tag == "utama":
        hasil_lapisan += baris
        detail_titik[pid] = {"rms": rms_pct, "faktor_rho": faktor_rho, "n_data": len(obs), "iter": it}
    else:
        hasil_verif.append({"point_id": pid, "NL": NL, "baris": baris, "rms": rms_pct})
    print("Titik %s | %d lapisan | %-10s | RMS = %.2f %% | iterasi = %d" % (pid, NL, tag, rms_pct, it))

# titik yang tidak terinversi tetap dicatat (jumlah titik output = input)
sudah = set([r["point_id"] for r in hasil_lapisan])
for pid in titik:
    if pid not in sudah:
        hasil_lapisan.append({"point_id": pid, "layer_no": np.nan, "depth_top_m": np.nan,
                              "depth_bottom_m": np.nan, "thickness_m": np.nan,
                              "resistivity_ohm_m": np.nan, "rms_pct": np.nan})
df_lay = pd.DataFrame(hasil_lapisan)
df_lay.to_csv(os.path.join(OUT_DIR, "ves_layers.csv"), index=False)

# ----------------------------- INTERPRETASI (Markdown) -----------------------
borelog = None
if os.path.exists(PATH_BORELOG):
    borelog = pd.read_csv(PATH_BORELOG)

tabel_lito = [(0, 10, "lempung / material jenuh air asin (indikasi)"),
              (10, 50, "lempung pasiran atau pasir lempungan jenuh (indikasi)"),
              (50, 200, "pasir-kerikil jenuh air tawar (indikasi)"),
              (200, 1e12, "material kering / batuan kompak (indikasi)")]

md = []
md.append("# Interpretasi VES Schlumberger\n")
md.append("Lokasi: %s | Litologi dominan: %s\n" % (LOKASI, LITOLOGI))
md.append("> Hasil ini indikasi awal untuk diskusi. **Bukan** keputusan titik/kedalaman bor final.\n")
md.append("## 1. Fakta terukur (data)\n")
md.append("- Jumlah titik input: %d; jumlah baris data: %d." % (len(titik), len(df)))
for q in sorted(set([x for f_ in flags for x in f_])):
    md.append("- Baris dengan flag `%s`: %d" % (q, sum([1 for f_ in flags if q in f_])))
md.append("- Titik tanpa flag: %d baris `ok`.\n" % sum([1 for f_ in flags if len(f_) == 0]))
md.append("## 2. Hasil inversi per titik (RMS dan lapisan)\n")
kandidat = []
for pid in titik:
    sub = df_lay[df_lay["point_id"] == pid]
    md.append("### Titik %s" % pid)
    if sub["layer_no"].isna().all():
        md.append("Tidak diinversi (data valid tidak cukup).\n"); continue
    md.append("RMS inversi = **%.2f %%** (n data = %d, iterasi = %d)\n" %
              (detail_titik[pid]["rms"], detail_titik[pid]["n_data"], detail_titik[pid]["iter"]))
    md.append("| Lapisan | Kedalaman (m) | Tebal (m) | rho (ohm.m) | faktor ketidakpastian rho | Dugaan litologi |")
    md.append("|---|---|---|---|---|---|")
    for _, r in sub.iterrows():
        lito = [t[2] for t in tabel_lito if t[0] <= r["resistivity_ohm_m"] < t[1]][0]
        kd = "%.1f - %s" % (r["depth_top_m"], ("%.1f" % r["depth_bottom_m"]) if not np.isnan(r["depth_bottom_m"]) else "tak hingga")
        tb = ("%.1f" % r["thickness_m"]) if not np.isnan(r["thickness_m"]) else "-"
        fk = detail_titik[pid]["faktor_rho"][int(r["layer_no"]) - 1]
        md.append("| %d | %s | %s | %.1f | x/÷ %.2f | %s |" % (r["layer_no"], kd, tb, r["resistivity_ohm_m"], fk, lito))
        if r["layer_no"] > 1 and AQ_RHO[0] <= r["resistivity_ohm_m"] <= AQ_RHO[1]:
            kandidat.append((pid, int(r["layer_no"]), r["depth_top_m"], r["depth_bottom_m"], r["resistivity_ohm_m"]))
    md.append("")
md.append("## 3. Kandidat lapisan akuifer (dugaan)\n")
md.append("Kriteria: lapisan bukan lapisan teratas dengan rho %.0f-%.0f ohm.m (asumsi, bandingkan rentang umum pasir jenuh vs lempung).\n" % AQ_RHO)
if len(kandidat) == 0:
    md.append("Tidak ada lapisan yang memenuhi kriteria.\n")
for k in kandidat:
    bt = "tak hingga (ketebalan tak terjamin)" if np.isnan(k[3]) else "%.1f m" % k[3]
    md.append("- Titik %s, lapisan %d: kedalaman %.1f - %s, rho = %.1f ohm.m" % (k[0], k[1], k[2], bt, k[4]))
    if borelog is not None:
        md.append("  - (pembanding borelog tersedia: periksa manual litologi pada rentang kedalaman ini)")
md.append("")
md.append("## 4. Asumsi dan keterbatasan\n")
md.append("- Model %d lapisan, model awal dari kuantil data, batas rho %.0f-%.0f ohm.m, tebal min %.1f m." % (N_LAYERS, RHO_MIN, RHO_MAX, H_MIN))
md.append("- **Prinsip ekuivalensi**: kombinasi tebal dan rho berbeda dapat memberi kurva yang hampir sama; kolom faktor ketidakpastian hanya perkiraan linear.")
md.append("- Lapisan terbawah dianggap half-space; ketebalannya tidak terukur.")
md.append("- Rentang rho untuk litologi bersifat umum dan tumpang tindih; wajib dikorelasikan dengan data bor.")
md.append("- Grafik (kurva sounding dan model inversi) hanya direncanakan, belum dibuat.\n")
if len(hasil_verif) > 0:
    md.append("## 5. Uji kestabilan jumlah lapisan (3 vs 4 lapisan, 2 titik)\n")
    for v in hasil_verif:
        s = "; ".join(["L%d rho=%.0f top=%.1f" % (b["layer_no"], b["resistivity_ohm_m"], b["depth_top_m"]) for b in v["baris"]])
        md.append("- Titik %s, %d lapisan, RMS %.2f %%: %s" % (v["point_id"], v["NL"], v["rms"], s))
    md.append("\nBandingkan manual: bila kedalaman akuifer utama berbeda jauh antar jumlah lapisan, laporkan model **tidak stabil**.\n")
f = open(os.path.join(OUT_DIR, "interpretation.md"), "w", encoding="utf-8"); f.write("\n".join(md)); f.close()

# ----------------------------- CEK SHA-256 SESUDAH ---------------------------
f = open(PATH_VES, "rb"); sha_sesudah = hashlib.sha256(f.read()).hexdigest(); f.close()
if sha_sebelum != sha_sesudah:
    sys.exit("GAGAL: ves_points.csv berubah selama proses!")

# ----------------------------- RUN CONFIG ------------------------------------
cfg = {"n_layers": N_LAYERS, "rho_min": RHO_MIN, "rho_max": RHO_MAX, "h_min_m": H_MIN,
       "jump_log10": JUMP_LOG10, "aquifer_rho_range": AQ_RHO, "max_iter": MAX_ITER,
       "model_awal": "rho: rata-rata log10 data per kelompok AB/2; tebal: deret geometri",
       "metode": "Levenberg-Marquardt (log10), forward Pekeris + integral Hankel J1",
       "sha256_ves_points": sha_sesudah, "sha256_identik": sha_sebelum == sha_sesudah,
       "versi": {"python": platform.python_version(), "numpy": np.__version__,
                 "scipy": scipy.__version__, "pandas": pd.__version__}}
f = open(os.path.join(OUT_DIR, "run_config.json"), "w"); json.dump(cfg, f, indent=2); f.close()
print("SHA-256 identik:", sha_sebelum == sha_sesudah)
print("Selesai. Titik input = %d, titik output = %d" % (len(titik), df_lay["point_id"].nunique()))

In [4]:
METHOD = "REFRACTION"  # pilih: VES / REFRACTION / GRAVITY / SEISMICITY / MT
BASE_DIR = Path(".")
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd

folder = BASE_DIR / "data" / "raw"
folder.mkdir(parents=True, exist_ok=True)

V1, V2, h1 = 400.0, 1800.0, 5.0
ti_ms = 2 * h1 * np.sqrt(V2**2 - V1**2) / (V1 * V2) * 1000.0
rx = np.arange(2.0, 61.0, 2.0)          # geophone tiap 2 m
rng = np.random.default_rng(0)

baris = []
# shot forward di x = 0 m
for x in rx:
    t = min(x / V1 * 1000.0, x / V2 * 1000.0 + ti_ms) + rng.normal(0, 0.3)
    baris.append(["S1", 0.0, x, round(t, 2), "forward"])
# shot reverse di x = 62 m
for x in rx:
    off = 62.0 - x
    t = min(off / V1 * 1000.0, off / V2 * 1000.0 + ti_ms) + rng.normal(0, 0.3)
    baris.append(["S2", 62.0, x, round(t, 2), "reverse"])

pd.DataFrame(baris, columns=["shot_id", "shot_x_m", "receiver_x_m", "t_fb_ms", "direction"]).to_csv(
    folder / "refraction_picks.csv", index=False)
print("File dibuat di:", (folder / "refraction_picks.csv").resolve())

# SEISMIK REFRAKSI 
if METHOD.upper() == "REFRACTION":
    INPUT_FILE = BASE_DIR / "data" / "raw" / "refraction_picks.csv"
    OUT_DIR = RESULTS_DIR / "refraction"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Offset breakpoint manual dalam meter; isi sesuai data dan inspeksi kurva.
    BREAKPOINT_M = 20.0
    VELOCITY_DIFF_FLAG_PCT = 15.0
    GEO_PHONE_SPACING_M = np.nan
    N_GEOPHONES = np.nan

    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"File input tidak ditemukan: {INPUT_FILE.resolve()}")
    SHA_BEFORE = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
    df = pd.read_csv(INPUT_FILE)
    required = ["shot_id", "shot_x_m", "receiver_x_m", "t_fb_ms", "direction"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Kolom refraksi tidak ditemukan: {missing}")
    df["direction"] = df["direction"].astype(str).str.lower().str.strip()
    if not set(df["direction"].dropna().unique()).issubset({"forward", "reverse"}):
        raise ValueError("Kolom direction hanya boleh berisi forward/reverse.")
    if df[required].isna().any().any():
        raise ValueError("Ada nilai kosong pada kolom wajib; periksa input tanpa menghapus baris.")

    df["offset_m"] = (df["receiver_x_m"] - df["shot_x_m"]).abs()
    df["t_s"] = df["t_fb_ms"] / 1000.0
    fit_rows = []
    depth_rows = []
    used_indices = set()
    unused_notes = []

    for direction in ["forward", "reverse"]:
        part = df[df["direction"] == direction].copy().sort_values("offset_m")
        if len(part) < 4:
            print(f"Peringatan: titik {direction} kurang dari 4; arah dilewati.")
            continue
        direct = part[part["offset_m"] <= BREAKPOINT_M]
        refracted = part[part["offset_m"] > BREAKPOINT_M]
        if len(direct) < 2 or len(refracted) < 2:
            raise ValueError(
                f"Breakpoint {BREAKPOINT_M} m tidak menghasilkan minimal 2 titik per segmen untuk {direction}. "
                "Ubah BREAKPOINT_M secara eksplisit."
            )

        # Regresi waktu (s) terhadap offset (m); kemiringan = slowness (s/m)
        direct_fit = np.polyfit(direct["offset_m"], direct["t_s"], 1)
        refr_fit = np.polyfit(refracted["offset_m"], refracted["t_s"], 1)
        direct_pred = np.polyval(direct_fit, direct["offset_m"])
        refr_pred = np.polyval(refr_fit, refracted["offset_m"])
        r2_direct = 1 - np.sum((direct["t_s"] - direct_pred)**2) / np.sum((direct["t_s"] - direct["t_s"].mean())**2)
        r2_refr = 1 - np.sum((refracted["t_s"] - refr_pred)**2) / np.sum((refracted["t_s"] - refracted["t_s"].mean())**2)

        slope1 = direct_fit[0]
        slope2 = refr_fit[0]
        if slope1 <= 0 or slope2 <= 0:
            raise ValueError(f"Kemiringan regresi tidak positif pada {direction}; periksa picks/breakpoint.")
        V1 = 1.0 / slope1
        V2 = 1.0 / slope2
        if V2 <= V1:
            raise ValueError(f"V2 <= V1 pada {direction}; rumus intercept-time tidak valid.")
        intercept_s = refr_fit[1]
        crossover = max(0.0, intercept_s / (slope1 - slope2)) if not np.isclose(slope1, slope2) else np.nan
        h1 = (intercept_s * V1 * V2) / (2.0 * np.sqrt(V2**2 - V1**2))

        fit_rows.append({
            "direction": direction, "V1_m_s": V1, "V2_m_s": V2,
            "intercept_ms": intercept_s * 1000, "crossover_m": crossover,
            "r2_direct": r2_direct, "r2_refracted": r2_refr,
            "n_direct": len(direct), "n_refracted": len(refracted)
        })
        depth_rows.append({
            "direction": direction, "h1_m": h1, "V_ratio_V2_V1": V2 / V1,
            "flag": "OK" if h1 >= 0 else "CHECK_NEGATIVE_DEPTH"
        })
        used_indices.update(direct.index.tolist() + refracted.index.tolist())

    fit_df = pd.DataFrame(fit_rows)
    depth_df = pd.DataFrame(depth_rows)
    if len(fit_df) == 2:
        v2f = fit_df.loc[fit_df["direction"] == "forward", "V2_m_s"].iloc[0]
        v2r = fit_df.loc[fit_df["direction"] == "reverse", "V2_m_s"].iloc[0]
        difference_pct = abs(v2f - v2r) / ((v2f + v2r) / 2) * 100
        if difference_pct > VELOCITY_DIFF_FLAG_PCT:
            depth_df["flag"] = "INDIKASI_BIDANG_MIRING_ATAU_HETEROGEN"
    else:
        difference_pct = np.nan

    fit_df.to_csv(OUT_DIR / "velocity_fit.csv", index=False)
    depth_df.to_csv(OUT_DIR / "depth_estimate.csv", index=False)
    unused = df.loc[~df.index.isin(used_indices)].copy()
    unused["note"] = "Tidak masuk regresi; periksa arah shot, breakpoint, atau data input."
    unused.to_csv(OUT_DIR / "picks_not_used.csv", index=False)

    SHA_AFTER = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
    config = {
        "input": str(INPUT_FILE), "breakpoint_method": "manual offset",
        "breakpoint_m": BREAKPOINT_M, "velocity_difference_flag_pct": VELOCITY_DIFF_FLAG_PCT,
        "formula_h1": "h1=(ti*V1*V2)/(2*sqrt(V2^2-V1^2))",
        "python_version": sys.version, "numpy_version": np.__version__, "pandas_version": pd.__version__,
        "sha256_before": SHA_BEFORE, "sha256_after": SHA_AFTER, "input_unchanged": SHA_BEFORE == SHA_AFTER
    }
    (OUT_DIR / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    report = f"""# Laporan Seismik Refraksi

- Breakpoint manual: {BREAKPOINT_M} m.
- Perbedaan relatif V2 forward–reverse: {difference_pct if np.isfinite(difference_pct) else 'tidak dapat dihitung'} %.
- Ambang flag perbedaan V2: {VELOCITY_DIFF_FLAG_PCT}%.
- Picks yang tidak masuk regresi: {len(unused)}; lihat `picks_not_used.csv`.
- SHA-256 input sebelum/sesudah sama: {SHA_BEFORE == SHA_AFTER}.
- Kedalaman h1 adalah estimasi berdasarkan model dua lapisan dengan bidang bias datar.
- Jika V2 forward dan reverse berbeda besar, hal tersebut dapat mengindikasikan kemiringan bidang atau heterogenitas; jangan dirata-ratakan tanpa evaluasi.
"""
    (OUT_DIR / "report.md").write_text(report, encoding="utf-8")
    print("Seismik refraksi selesai:", OUT_DIR.resolve())
    print(METHOD)

File dibuat di: D:\Semester 5\Metode AI\data\raw\refraction_picks.csv
Seismik refraksi selesai: D:\Semester 5\Metode AI\results\refraction
REFRACTION


---

## Prompt 3 — Gravitasi: Filter Derivatif (FHD dan SVD) untuk Delineasi Struktur

### [R] Role
Bertindaklah sebagai reviewer implementasi numerik untuk pengolahan data potensial. Fokus pada satuan turunan, efek tepi grid, dan pemisahan antara hasil filter dan interpretasi geologi.

### [T] Task
Hitung First Horizontal Derivative (FHD) dan Second Vertical Derivative (SVD) dari grid anomali Bouguer `data/processed/bouguer_grid.tif`, lalu ekstrak titik kandidat tepi anomali dari maksimum FHD dan kontur nol SVD. Tuliskan hasil sebagai tabel kandidat, bukan sebagai peta patahan.

### [C] Context
Survei gravitasi regional untuk pemetaan awal batas cekungan dan struktur. Hasil filter dipakai tim interpretasi sebagai bahan diskusi awal tentang kemenerusan lineament. Hasil tidak dipakai untuk menetapkan patahan, estimasi kedalaman, atau pemodelan inversi.

### [I] Input
- `data/processed/bouguer_grid.tif`: GeoTIFF, anomali Bouguer dalam mGal, CRS `{{EPSG:... UTM}}` (satuan meter), ukuran sel `{{m}}`, nodata `{{-9999}}`.
- Rumus: `FHD = √((∂g/∂x)² + (∂g/∂y)²)`; `SVD = −(∂²g/∂x² + ∂²g/∂y²)` (asumsi medan memenuhi persamaan Laplace).
- Metode turunan: `{{beda hingga / FFT}}`.

### [K] Constraints
- Jangan mengubah file grid asli.
- Jangan melakukan upward continuation, penghalusan, atau pemisahan regional–residual, kecuali dinyatakan pada Input.
- Sel nodata harus tetap nodata pada output; jangan diisi interpolasi.
- Jika CRS bukan sistem koordinat metrik, berhenti dan tanyakan (satuan turunan menjadi tidak valid).
- Tepi grid (`{{n}}` sel) diberi flag efek tepi.
- Jangan menyimpulkan keberadaan patahan atau kedalaman sumber.
- Software: Python dengan rasterio, NumPy, SciPy; tanpa jaringan.

### [O] Output (rencana teks)
- `results/gravity/fhd_mgal_per_m.tif` dan `results/gravity/svd_mgal_per_m2.tif`: grid dengan CRS, transform, dan nodata identik dengan input.
- `results/gravity/edge_candidates.csv`: `id`, `x_m`, `y_m`, `fhd_value`, `svd_value`, `edge_flag`.
- `results/gravity/report.md`: narasi statistik FHD dan SVD, jumlah kandidat, sel yang di-flag, dan pernyataan bahwa hasil belum diinterpretasi secara geologi.
- `results/gravity/run_config.json`: rumus, metode turunan, ukuran sel, versi package.

### [Q] Quality criteria
- Shape, CRS, dan transform output sama dengan input.
- Satuan mGal/m dan mGal/m² eksplisit.
- Sel nodata input tetap nodata pada output.
- Kriteria pemilihan kandidat (ambang FHD, toleransi SVD ≈ 0) tercatat.

### [V] Optional verification
- Check: ramp linear sintetis `g = a·x`. Expected: FHD konstan (= `a`) dan SVD ≈ 0. Jika gagal: laporkan bug pada perhitungan turunan.
- Check: grid sintetis berupa anomali bola/silinder. Expected: maksimum FHD berada di sekitar tepi anomali. Jika tidak: periksa ukuran sel dan orientasi sumbu.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

In [8]:
%pip install rasterio

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os, sys, json, hashlib, importlib.util
from pathlib import Path

# ---- 1) arahkan PROJ ke data milik rasterio (sebelum rasterio di-import) ----
spec = importlib.util.find_spec("rasterio")
proj_dir = os.path.join(os.path.dirname(spec.origin), "proj_data")
if os.path.isdir(proj_dir):
    os.environ["PROJ_DATA"] = proj_dir
    os.environ["PROJ_LIB"] = proj_dir

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from scipy.ndimage import maximum_filter

# ---- 2) pengaturan ----
try:
    BASE_DIR
except NameError:
    BASE_DIR = Path.cwd()
BASE_DIR = Path(BASE_DIR)
try:
    RESULTS_DIR
except NameError:
    RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR = Path(RESULTS_DIR)

INPUT_FILE = BASE_DIR / "data" / "processed" / "bouguer_grid.tif"
OUT_DIR = RESULTS_DIR / "gravity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

METODE = "beda hingga"       # "beda hingga" atau "FFT"
N_EDGE_CELLS = 2             # lebar pita tepi grid yang di-flag
FHD_PERSENTIL = 90           # ambang FHD kandidat (persentil)
SVD_TOL_FRAC = 0.10          # |SVD| <= frac * std(SVD) dianggap ~ nol

UTM50S_WKT = ('PROJCS["WGS 84 / UTM zone 50S",GEOGCS["WGS 84",DATUM["WGS_1984",'
              'SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],'
              'UNIT["degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],'
              'PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",117],'
              'PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],'
              'PARAMETER["false_northing",10000000],UNIT["metre",1]]')

# ---- 3) buat grid sintetis bila file belum ada ----
BUAT_ULANG_SINTETIK = False   # True = timpa bouguer_grid.tif dengan grid SINTETIS (JANGAN untuk data asli)
if BUAT_ULANG_SINTETIK or not INPUT_FILE.exists():
    INPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    n, dx_s = 100, 100.0
    x0, y0 = 765000.0, 9436000.0
    xs = x0 + (np.arange(n) + 0.5) * dx_s
    ys = y0 - (np.arange(n) + 0.5) * dx_s
    X, Y = np.meshgrid(xs, ys)
    # anomali "kontak" (tepi cekungan): g = A * (arctan((x - xc)/w)/pi + 0.5)
    # xc di tengah sel kolom ke-50 agar puncak FHD dan nol SVD jatuh tepat pada pusat sel
    xc = x0 + 50.5 * dx_s
    lebar = 500.0
    g_s = 10.0 * (np.arctan((X - xc) / lebar) / np.pi + 0.5)
    g_s[:5, :5] = -9999.0
    try:
        crs_tulis = rasterio.crs.CRS.from_epsg(32750)
    except Exception:
        crs_tulis = rasterio.crs.CRS.from_wkt(UTM50S_WKT)
    with rasterio.open(INPUT_FILE, "w", driver="GTiff", height=n, width=n, count=1,
                       dtype="float32", crs=crs_tulis, transform=from_origin(x0, y0, dx_s, dx_s),
                       nodata=-9999.0) as dst:
        dst.write(g_s.astype("float32"), 1)
    print("Grid SINTETIS dibuat di:", INPUT_FILE.resolve())

# ---- 4) baca grid ----
SHA_BEFORE = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
with rasterio.open(INPUT_FILE) as src:
    g_raw = src.read(1).astype("float64")
    profile = src.profile.copy()
    transform = src.transform
    crs = src.crs
    nodata_in = src.nodata

galat_crs = None
try:
    metrik = crs is not None and crs.is_projected
except Exception as e:
    galat_crs = repr(e)[:200]
    metrik = abs(transform.a) > 1.0     # cadangan: ukuran sel > 1 satuan -> bukan derajat
if not metrik:
    print("DIAGNOSA -> file:", INPUT_FILE.resolve())
    print("DIAGNOSA -> CRS terbaca:", crs)
    print("DIAGNOSA -> transform:", transform)
    print("DIAGNOSA -> ukuran sel:", abs(transform.a), "x", abs(transform.e))
    print("DIAGNOSA -> galat PROJ:", galat_crs)
    raise ValueError(
        "CRS/ukuran sel tidak valid untuk turunan metrik. Jika ini file SINTETIS sisa percobaan gagal, "
        "hapus bouguer_grid.tif lalu jalankan ulang (atau set BUAT_ULANG_SINTETIK = True). "
        "Jika ini data asli, periksa CRS-nya.")

dx, dy = abs(transform.a), abs(transform.e)
mask_nodata = ~np.isfinite(g_raw)
if nodata_in is not None:
    mask_nodata |= (g_raw == nodata_in)
g = g_raw.copy()
g[mask_nodata] = np.nan
ny, nx = g.shape

# ---- 5) turunan ----
if METODE == "FFT":
    g_f = np.where(mask_nodata, np.nanmean(g), g)
    kx = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(ny, d=dy)
    KX, KY = np.meshgrid(kx, ky)
    G = np.fft.fft2(g_f)
    gx = np.real(np.fft.ifft2(1j * KX * G))
    gy = np.real(np.fft.ifft2(1j * KY * G))
    gxx = np.real(np.fft.ifft2(-(KX**2) * G))
    gyy = np.real(np.fft.ifft2(-(KY**2) * G))
    for arr in (gx, gy, gxx, gyy):
        arr[mask_nodata] = np.nan
else:
    gx = np.full_like(g, np.nan); gy = np.full_like(g, np.nan)
    gxx = np.full_like(g, np.nan); gyy = np.full_like(g, np.nan)
    gx[:, 1:-1] = (g[:, 2:] - g[:, :-2]) / (2 * dx)
    gy[1:-1, :] = (g[2:, :] - g[:-2, :]) / (2 * dy)
    gxx[:, 1:-1] = (g[:, 2:] - 2 * g[:, 1:-1] + g[:, :-2]) / dx**2
    gyy[1:-1, :] = (g[2:, :] - 2 * g[1:-1, :] + g[:-2, :]) / dy**2

FHD = np.sqrt(gx**2 + gy**2)       # mGal/m
SVD = -(gxx + gyy)                 # mGal/m^2
mask_out = mask_nodata | ~np.isfinite(FHD) | ~np.isfinite(SVD)
FHD[mask_out] = np.nan
SVD[mask_out] = np.nan

# ---- 6) tulis GeoTIFF ----
nodata_tulis = nodata_in if nodata_in is not None else -9999.0
profile.update(driver="GTiff", dtype="float32", count=1, nodata=nodata_tulis)
with rasterio.open(OUT_DIR / "fhd_mgal_per_m.tif", "w", **profile) as dst:
    dst.write(np.where(np.isnan(FHD), nodata_tulis, FHD).astype("float32"), 1)
with rasterio.open(OUT_DIR / "svd_mgal_per_m2.tif", "w", **profile) as dst:
    dst.write(np.where(np.isnan(SVD), nodata_tulis, SVD).astype("float32"), 1)

# ---- 7) kandidat tepi ----
valid = ~mask_out
thr_fhd = np.nanpercentile(FHD[valid], FHD_PERSENTIL)
tol_svd = SVD_TOL_FRAC * np.nanstd(SVD[valid])
FHD_f = np.where(valid, FHD, -np.inf)
is_max = (FHD_f == maximum_filter(FHD_f, size=3)) & valid
cand = is_max & (FHD >= thr_fhd) & (np.abs(SVD) <= tol_svd)

edge_band = np.zeros(g.shape, dtype=bool)
edge_band[:N_EDGE_CELLS, :] = True; edge_band[-N_EDGE_CELLS:, :] = True
edge_band[:, :N_EDGE_CELLS] = True; edge_band[:, -N_EDGE_CELLS:] = True

rr, cc = np.where(cand)
px = transform.c + (cc + 0.5) * transform.a
py = transform.f + (rr + 0.5) * transform.e
tab = pd.DataFrame({"id": np.arange(1, len(rr) + 1), "x_m": px, "y_m": py,
                    "fhd_value": FHD[rr, cc], "svd_value": SVD[rr, cc],
                    "edge_flag": edge_band[rr, cc]})
tab.to_csv(OUT_DIR / "edge_candidates.csv", index=False)

# ---- 8) verifikasi ----
a_ramp = 0.002
g_ramp = np.tile(a_ramp * np.arange(60) * 10.0, (50, 1))
gx_r = (g_ramp[:, 2:] - g_ramp[:, :-2]) / 20.0
gxx_r = (g_ramp[:, 2:] - 2 * g_ramp[:, 1:-1] + g_ramp[:, :-2]) / 100.0
cek_ramp = bool(np.allclose(gx_r, a_ramp, rtol=1e-6) and np.allclose(gxx_r, 0, atol=1e-9))

SHA_AFTER = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
with rasterio.open(OUT_DIR / "fhd_mgal_per_m.tif") as chk:
    cek_grid = (chk.shape == (ny, nx)) and (chk.transform == transform)

# ---- 9) laporan & config ----
report = f"""# Laporan Gravitasi - FHD dan SVD

- Grid: {ny} x {nx} sel, ukuran sel {dx:.2f} x {dy:.2f} m.
- Sel nodata input: {int(mask_nodata.sum())}; sel nodata output: {int(mask_out.sum())}.
- FHD (mGal/m): min {np.nanmin(FHD):.3e}, maks {np.nanmax(FHD):.3e}, rata-rata {np.nanmean(FHD):.3e}.
- SVD (mGal/m^2): min {np.nanmin(SVD):.3e}, maks {np.nanmax(SVD):.3e}, std {np.nanstd(SVD):.3e}.
- Kriteria kandidat: maksimum lokal FHD (3x3) >= persentil ke-{FHD_PERSENTIL} ({thr_fhd:.3e}) dan |SVD| <= {SVD_TOL_FRAC} x std ({tol_svd:.3e}).
- Jumlah kandidat: {len(tab)}; di-flag efek tepi: {int(tab['edge_flag'].sum())}.
- Metode turunan: {METODE}.
- Hasil BELUM diinterpretasi secara geologi; bukan peta patahan dan bukan estimasi kedalaman.
- Cek ramp linear: {'LULUS' if cek_ramp else 'GAGAL'}; cek shape/transform output: {'LULUS' if cek_grid else 'GAGAL'}.
- SHA-256 input sebelum/sesudah sama: {SHA_BEFORE == SHA_AFTER}.
"""
(OUT_DIR / "report.md").write_text(report, encoding="utf-8")

config = {"input": str(INPUT_FILE), "metode_turunan": METODE, "ukuran_sel_m": [dx, dy],
          "n_edge_cells": N_EDGE_CELLS, "fhd_persentil": FHD_PERSENTIL, "svd_tol_frac_std": SVD_TOL_FRAC,
          "formula": {"FHD": "sqrt(gx^2+gy^2)", "SVD": "-(gxx+gyy)"},
          "python": sys.version, "numpy": np.__version__, "pandas": pd.__version__,
          "rasterio": rasterio.__version__, "sha256_before": SHA_BEFORE, "sha256_after": SHA_AFTER,
          "input_unchanged": SHA_BEFORE == SHA_AFTER}
(OUT_DIR / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print("Gravitasi selesai. Kandidat:", len(tab), "| ramp:", cek_ramp, "| grid:", cek_grid, "| hash sama:", SHA_BEFORE == SHA_AFTER)
print("Hasil di:", OUT_DIR.resolve())

Grid SINTETIS dibuat di: D:\Semester 5\Metode AI\data\processed\bouguer_grid.tif
Gravitasi selesai. Kandidat: 98 | ramp: True | grid: True | hash sama: True
Hasil di: D:\Semester 5\Metode AI\results\gravity


---

## Prompt 4 — Seismologi: Magnitude of Completeness (Mc) dan b-value Gutenberg-Richter

### [R] Role
Bertindaklah sebagai reviewer analisis statistik seismisitas. Fokus pada konsistensi tipe magnitudo, pemilihan Mc, dan ketidakpastian estimasi b-value.

### [T] Task
Dari katalog gempa yang telah dinormalisasi `data/processed/events.csv`, hitung frequency–magnitude distribution (FMD), tentukan Mc dengan metode maximum curvature, lalu estimasi b-value dengan maximum likelihood (Aki–Utsu) beserta ketidakpastiannya untuk event dengan M ≥ Mc. Jangan menyimpulkan tingkat bahaya gempa.

### [C] Context
Analisis awal seismisitas satu wilayah tektonik untuk eksplorasi data. Hasil dipakai sebagai gambaran kelengkapan katalog dan karakter statistik awal. Bukan seismic-hazard assessment dan bukan prediksi gempa.

### [I] Input
- `data/processed/events.csv`: `event_id`, `origin_time_utc`, `latitude`, `longitude`, `depth_km`, `magnitude`, `magnitude_type`, `agency`.
- Tipe magnitudo yang dipakai: `{{ML/mb/Mw}}` (hanya satu tipe). Bin magnitudo ΔM = `{{0.1}}`.
- Rumus: `b = log10(e) / (mean(M) − (Mc − ΔM/2))`; `σ_b = 2.30 · b² · √(Σ(Mi − mean(M))² / (N(N−1)))`.
- Batas minimum jumlah event untuk estimasi: `{{50}}`.

### [K] Constraints
- Jangan mengubah `data/processed/events.csv`.
- Jangan mencampur atau mengonversi tipe magnitudo; event dengan tipe lain dikecualikan dan dicatat jumlahnya.
- Jangan melakukan declustering kecuali dinyatakan pada Input.
- Jika jumlah event di atas Mc kurang dari batas minimum, jangan menghitung b-value: laporkan dan berhenti.
- Jangan menafsirkan nilai b terhadap tegangan, fluida, atau hazard.
- Software: Python, NumPy, pandas; tanpa jaringan.

### [O] Output (rencana teks)
- `results/seismicity/fmd.csv`: `mag_bin`, `n_incremental`, `n_cumulative`.
- `results/seismicity/bvalue_summary.json`: `mc`, `b_value`, `sigma_b`, `n_above_mc`, `magnitude_type`, `time_range`.
- `results/seismicity/report.md`: narasi jumlah event total dan yang dikecualikan, Mc, b-value dengan ketidakpastian, serta catatan keterbatasan (maximum curvature cenderung mengestimasi Mc terlalu rendah).
- `results/seismicity/run_config.json`: ΔM, metode Mc, rumus, versi package.

### [Q] Quality criteria
- Jumlah event yang dipakai + dikecualikan = jumlah baris input.
- b-value selalu dilaporkan bersama σ_b, Mc, dan N.
- Tipe magnitudo dan periode waktu dinyatakan eksplisit.
- Tidak ada interpretasi tektonik atau hazard pada laporan.

### [V] Optional verification
- Check: katalog sintetis dengan b = 1,0 (distribusi eksponensial magnitudo, Mc = 2,0, N = 1000). Expected: b terpulihkan dalam ± `{{0,1}}`. Jika gagal: laporkan bug.
- Check: bandingkan Mc pada ΔM = 0,1 dan 0,2. Expected: selisih dilaporkan sebagai sensitivitas. Jika besar: tandai Mc tidak stabil.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

In [10]:
import os, sys, json, hashlib, importlib.util
from pathlib import Path

# ---- 1) arahkan PROJ ke data milik rasterio (sebelum rasterio di-import) ----
spec = importlib.util.find_spec("rasterio")
proj_dir = os.path.join(os.path.dirname(spec.origin), "proj_data")
if os.path.isdir(proj_dir):
    os.environ["PROJ_DATA"] = proj_dir
    os.environ["PROJ_LIB"] = proj_dir

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from scipy.ndimage import maximum_filter

# ---- 2) pengaturan ----
BASE_DIR = Path.cwd()     # otomatis = folder kerja notebook; ganti manual bila perlu, mis. Path(r"D:\Tugas\geofisika")
RESULTS_DIR = BASE_DIR / "results"
print("Folder kerja (BASE_DIR):", BASE_DIR.resolve())

INPUT_FILE = BASE_DIR / "data" / "processed" / "bouguer_grid.tif"
OUT_DIR = RESULTS_DIR / "gravity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

METODE = "beda hingga"       # "beda hingga" atau "FFT"
N_EDGE_CELLS = 2             # lebar pita tepi grid yang di-flag
FHD_PERSENTIL = 90           # ambang FHD kandidat (persentil)
SVD_TOL_FRAC = 0.10          # |SVD| <= frac * std(SVD) dianggap ~ nol

UTM50S_WKT = ('PROJCS["WGS 84 / UTM zone 50S",GEOGCS["WGS 84",DATUM["WGS_1984",'
              'SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],'
              'UNIT["degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],'
              'PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",117],'
              'PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],'
              'PARAMETER["false_northing",10000000],UNIT["metre",1]]')

# ---- 3) buat grid sintetis bila file belum ada ----
BUAT_ULANG_SINTETIK = False   # True = timpa bouguer_grid.tif dengan grid SINTETIS (JANGAN untuk data asli)
if BUAT_ULANG_SINTETIK or not INPUT_FILE.exists():
    INPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    n, dx_s = 100, 100.0
    x0, y0 = 765000.0, 9436000.0
    xs = x0 + (np.arange(n) + 0.5) * dx_s
    ys = y0 - (np.arange(n) + 0.5) * dx_s
    X, Y = np.meshgrid(xs, ys)
    # anomali "kontak" (tepi cekungan): g = A * (arctan((x - xc)/w)/pi + 0.5)
    # xc di tengah sel kolom ke-50 agar puncak FHD dan nol SVD jatuh tepat pada pusat sel
    xc = x0 + 50.5 * dx_s
    lebar = 500.0
    g_s = 10.0 * (np.arctan((X - xc) / lebar) / np.pi + 0.5)
    g_s[:5, :5] = -9999.0
    try:
        crs_tulis = rasterio.crs.CRS.from_epsg(32750)
    except Exception:
        crs_tulis = rasterio.crs.CRS.from_wkt(UTM50S_WKT)
    with rasterio.open(INPUT_FILE, "w", driver="GTiff", height=n, width=n, count=1,
                       dtype="float32", crs=crs_tulis, transform=from_origin(x0, y0, dx_s, dx_s),
                       nodata=-9999.0) as dst:
        dst.write(g_s.astype("float32"), 1)
    print("Grid SINTETIS dibuat di:", INPUT_FILE.resolve())

# ---- 4) baca grid ----
SHA_BEFORE = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
with rasterio.open(INPUT_FILE) as src:
    g_raw = src.read(1).astype("float64")
    profile = src.profile.copy()
    transform = src.transform
    crs = src.crs
    nodata_in = src.nodata

galat_crs = None
try:
    metrik = crs is not None and crs.is_projected
except Exception as e:
    galat_crs = repr(e)[:200]
    metrik = abs(transform.a) > 1.0     # cadangan: ukuran sel > 1 satuan -> bukan derajat
if not metrik:
    print("DIAGNOSA -> file:", INPUT_FILE.resolve())
    print("DIAGNOSA -> CRS terbaca:", crs)
    print("DIAGNOSA -> transform:", transform)
    print("DIAGNOSA -> ukuran sel:", abs(transform.a), "x", abs(transform.e))
    print("DIAGNOSA -> galat PROJ:", galat_crs)
    raise ValueError(
        "CRS/ukuran sel tidak valid untuk turunan metrik. Jika ini file SINTETIS sisa percobaan gagal, "
        "hapus bouguer_grid.tif lalu jalankan ulang (atau set BUAT_ULANG_SINTETIK = True). "
        "Jika ini data asli, periksa CRS-nya.")

dx, dy = abs(transform.a), abs(transform.e)
mask_nodata = ~np.isfinite(g_raw)
if nodata_in is not None:
    mask_nodata |= (g_raw == nodata_in)
g = g_raw.copy()
g[mask_nodata] = np.nan
ny, nx = g.shape

# ---- 5) turunan ----
if METODE == "FFT":
    g_f = np.where(mask_nodata, np.nanmean(g), g)
    kx = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(ny, d=dy)
    KX, KY = np.meshgrid(kx, ky)
    G = np.fft.fft2(g_f)
    gx = np.real(np.fft.ifft2(1j * KX * G))
    gy = np.real(np.fft.ifft2(1j * KY * G))
    gxx = np.real(np.fft.ifft2(-(KX**2) * G))
    gyy = np.real(np.fft.ifft2(-(KY**2) * G))
    for arr in (gx, gy, gxx, gyy):
        arr[mask_nodata] = np.nan
else:
    gx = np.full_like(g, np.nan); gy = np.full_like(g, np.nan)
    gxx = np.full_like(g, np.nan); gyy = np.full_like(g, np.nan)
    gx[:, 1:-1] = (g[:, 2:] - g[:, :-2]) / (2 * dx)
    gy[1:-1, :] = (g[2:, :] - g[:-2, :]) / (2 * dy)
    gxx[:, 1:-1] = (g[:, 2:] - 2 * g[:, 1:-1] + g[:, :-2]) / dx**2
    gyy[1:-1, :] = (g[2:, :] - 2 * g[1:-1, :] + g[:-2, :]) / dy**2

FHD = np.sqrt(gx**2 + gy**2)       # mGal/m
SVD = -(gxx + gyy)                 # mGal/m^2
mask_out = mask_nodata | ~np.isfinite(FHD) | ~np.isfinite(SVD)
FHD[mask_out] = np.nan
SVD[mask_out] = np.nan

# ---- 6) tulis GeoTIFF ----
nodata_tulis = nodata_in if nodata_in is not None else -9999.0
profile.update(driver="GTiff", dtype="float32", count=1, nodata=nodata_tulis)
with rasterio.open(OUT_DIR / "fhd_mgal_per_m.tif", "w", **profile) as dst:
    dst.write(np.where(np.isnan(FHD), nodata_tulis, FHD).astype("float32"), 1)
with rasterio.open(OUT_DIR / "svd_mgal_per_m2.tif", "w", **profile) as dst:
    dst.write(np.where(np.isnan(SVD), nodata_tulis, SVD).astype("float32"), 1)

# ---- 7) kandidat tepi ----
valid = ~mask_out
thr_fhd = np.nanpercentile(FHD[valid], FHD_PERSENTIL)
tol_svd = SVD_TOL_FRAC * np.nanstd(SVD[valid])
FHD_f = np.where(valid, FHD, -np.inf)
is_max = (FHD_f == maximum_filter(FHD_f, size=3)) & valid
cand = is_max & (FHD >= thr_fhd) & (np.abs(SVD) <= tol_svd)

edge_band = np.zeros(g.shape, dtype=bool)
edge_band[:N_EDGE_CELLS, :] = True; edge_band[-N_EDGE_CELLS:, :] = True
edge_band[:, :N_EDGE_CELLS] = True; edge_band[:, -N_EDGE_CELLS:] = True

rr, cc = np.where(cand)
px = transform.c + (cc + 0.5) * transform.a
py = transform.f + (rr + 0.5) * transform.e
tab = pd.DataFrame({"id": np.arange(1, len(rr) + 1), "x_m": px, "y_m": py,
                    "fhd_value": FHD[rr, cc], "svd_value": SVD[rr, cc],
                    "edge_flag": edge_band[rr, cc]})
tab.to_csv(OUT_DIR / "edge_candidates.csv", index=False)

# ---- 8) verifikasi ----
a_ramp = 0.002
g_ramp = np.tile(a_ramp * np.arange(60) * 10.0, (50, 1))
gx_r = (g_ramp[:, 2:] - g_ramp[:, :-2]) / 20.0
gxx_r = (g_ramp[:, 2:] - 2 * g_ramp[:, 1:-1] + g_ramp[:, :-2]) / 100.0
cek_ramp = bool(np.allclose(gx_r, a_ramp, rtol=1e-6) and np.allclose(gxx_r, 0, atol=1e-9))

SHA_AFTER = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
with rasterio.open(OUT_DIR / "fhd_mgal_per_m.tif") as chk:
    cek_grid = (chk.shape == (ny, nx)) and (chk.transform == transform)

# ---- 9) laporan & config ----
report = f"""# Laporan Gravitasi - FHD dan SVD

- Grid: {ny} x {nx} sel, ukuran sel {dx:.2f} x {dy:.2f} m.
- Sel nodata input: {int(mask_nodata.sum())}; sel nodata output: {int(mask_out.sum())}.
- FHD (mGal/m): min {np.nanmin(FHD):.3e}, maks {np.nanmax(FHD):.3e}, rata-rata {np.nanmean(FHD):.3e}.
- SVD (mGal/m^2): min {np.nanmin(SVD):.3e}, maks {np.nanmax(SVD):.3e}, std {np.nanstd(SVD):.3e}.
- Kriteria kandidat: maksimum lokal FHD (3x3) >= persentil ke-{FHD_PERSENTIL} ({thr_fhd:.3e}) dan |SVD| <= {SVD_TOL_FRAC} x std ({tol_svd:.3e}).
- Jumlah kandidat: {len(tab)}; di-flag efek tepi: {int(tab['edge_flag'].sum())}.
- Metode turunan: {METODE}.
- Hasil BELUM diinterpretasi secara geologi; bukan peta patahan dan bukan estimasi kedalaman.
- Cek ramp linear: {'LULUS' if cek_ramp else 'GAGAL'}; cek shape/transform output: {'LULUS' if cek_grid else 'GAGAL'}.
- SHA-256 input sebelum/sesudah sama: {SHA_BEFORE == SHA_AFTER}.
"""
(OUT_DIR / "report.md").write_text(report, encoding="utf-8")

config = {"input": str(INPUT_FILE), "metode_turunan": METODE, "ukuran_sel_m": [dx, dy],
          "n_edge_cells": N_EDGE_CELLS, "fhd_persentil": FHD_PERSENTIL, "svd_tol_frac_std": SVD_TOL_FRAC,
          "formula": {"FHD": "sqrt(gx^2+gy^2)", "SVD": "-(gxx+gyy)"},
          "python": sys.version, "numpy": np.__version__, "pandas": pd.__version__,
          "rasterio": rasterio.__version__, "sha256_before": SHA_BEFORE, "sha256_after": SHA_AFTER,
          "input_unchanged": SHA_BEFORE == SHA_AFTER}
(OUT_DIR / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print("Gravitasi selesai. Kandidat:", len(tab), "| ramp:", cek_ramp, "| grid:", cek_grid, "| hash sama:", SHA_BEFORE == SHA_AFTER)
print("Hasil di:", OUT_DIR.resolve())

Folder kerja (BASE_DIR): D:\Semester 5\Metode AI
Gravitasi selesai. Kandidat: 98 | ramp: True | grid: True | hash sama: True
Hasil di: D:\Semester 5\Metode AI\results\gravity


---

## Prompt 5 — Magnetotellurik: QC Kurva Sounding dan Transformasi Niblett-Bostick

### [R] Role
Bertindaklah sebagai reviewer QC data magnetotellurik. Fokus pada kualitas per frekuensi, konsistensi resistivitas semu–fase, dan pemisahan transformasi awal dari inversi.

### [T] Task
Periksa kurva sounding MT `data/raw/mt_site01.csv` untuk menandai frekuensi dengan error besar, fase di luar rentang yang diharapkan, dan lompatan resistivitas semu antarfrekuensi, lalu hitung transformasi Niblett–Bostick (kedalaman dan resistivitas Bostick) untuk mode XY dan YX. Jangan menjalankan inversi 1D atau 2D.

### [C] Context
Survei MT untuk investigasi struktur resistivitas dalam pada area prospek geotermal. QC dilakukan sebelum data diserahkan ke tahap inversi. Hasil transformasi Bostick hanya gambaran awal, bukan model resistivitas final dan bukan interpretasi reservoir.

### [I] Input
- `data/raw/mt_site01.csv`: `freq_hz` (Hz), `rho_xy`, `rho_yx` (Ω·m), `phi_xy`, `phi_yx` (derajat), `rho_err_pct`, `phi_err_deg`. Konvensi fase: `{{semua fase di kuadran 0–90° / YX di kuadran ketiga}}`.
- Rumus: `D_B = √(ρa / (ω · μ0))`, dengan `ω = 2π·f`, `μ0 = 4π × 10⁻⁷ H/m`; `ρ_B = ρa · (π / (2·φ) − 1)`, dengan φ dalam radian.
- Ambang: `rho_err_pct > {{10}}`, `phi_err_deg > {{5}}`, lompatan ρa antarfrekuensi bersebelahan > `{{faktor}}`.

### [K] Constraints
- Jangan mengubah `data/raw/`; jangan menghapus frekuensi, hanya beri flag.
- Jangan menjalankan inversi, static-shift correction, atau smoothing.
- Konvensi fase harus dinyatakan; jika tidak jelas, berhenti dan tanyakan (bisa membuat ρ_B salah).
- Bila φ ≥ 90° atau φ ≤ 0°, transformasi Bostick tidak dihitung untuk frekuensi tersebut dan alasannya dicatat.
- Software: Python dengan NumPy dan pandas; tanpa jaringan.
- Jangan mengaitkan hasil dengan reservoir, zona clay cap, atau sumber panas.

### [O] Output (rencana teks)
- `results/mt/mt_site01_qc.csv`: seluruh kolom input + `qc_flag` + `qc_note`.
- `results/mt/mt_site01_bostick.csv`: `freq_hz`, `mode`, `depth_bostick_m`, `rho_bostick_ohm_m`, `status`.
- `results/mt/qc_report.md`: narasi jumlah frekuensi, jumlah flag per jenis, rentang frekuensi yang andal, dan frekuensi yang tidak dapat ditransformasi.
- `results/mt/run_config.json`: ambang, konvensi fase, konstanta, versi package.

### [Q] Quality criteria
- Jumlah baris `qc.csv` sama dengan jumlah baris input; urutan frekuensi tidak berubah.
- Semua ambang dan konvensi fase tercatat.
- Satuan Hz, Ω·m, derajat, dan m eksplisit pada semua kolom.
- Laporan memisahkan hasil terukur, asumsi, dan hal yang belum ditafsirkan.

### [V] Optional verification
- Check: setengah-ruang homogen sintetis (ρ = 100 Ω·m, φ = 45°). Expected: ρ_B ≈ 100 Ω·m di semua frekuensi dan `D_B` sesuai rumus. Jika gagal: periksa satuan fase dan konstanta.
- Check: hitung ulang `D_B` untuk 5 frekuensi acak secara independen. Expected: sama dengan `bostick.csv`. Jika gagal: periksa penggunaan ω dan μ0.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

In [11]:
import os, sys, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------- PENGATURAN ----------------
BASE_DIR = Path.cwd()     # otomatis = folder kerja notebook; ganti manual bila perlu, mis. Path(r"D:\Tugas\geofisika")
RESULTS_DIR = BASE_DIR / "results"
print("Folder kerja (BASE_DIR):", BASE_DIR.resolve())
BREAKPOINT_MODE = "otomatis"    # "otomatis" (minimum SSE dua garis) atau "manual"
BREAKPOINT_MANUAL_M = {"forward": 13.0, "reverse": 13.0}   # offset breakpoint (m) bila mode manual
MIN_PTS_SEG = 3                 # jumlah titik minimum tiap segmen
AMBANG_V2_PCT = 15.0            # beda V2 forward-reverse -> flag
TOL_SINTETIK_PCT = 5.0
BUAT_ULANG_SINTETIK = False     # True = timpa refraction_picks.csv dengan data SINTETIS (jangan untuk data asli)

INPUT_FILE = BASE_DIR / "data" / "raw" / "refraction_picks.csv"
OUT_DIR = RESULTS_DIR / "refraction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- picks sintetis (bila file belum ada) ----------------
if BUAT_ULANG_SINTETIK or not INPUT_FILE.exists():
    INPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    V1s, V2s, h1s = 400.0, 1800.0, 5.0
    ti_s = 2 * h1s * np.sqrt(V2s**2 - V1s**2) / (V1s * V2s) * 1000.0
    rx = np.arange(2.0, 61.0, 2.0)
    baris = []
    for sid, sx, arah in [("S1", 0.0, "forward"), ("S2", 62.0, "reverse")]:
        for x in rx:
            off = abs(sx - x)
            t = min(off / V1s * 1000.0, off / V2s * 1000.0 + ti_s) + rng.normal(0, 0.3)
            baris.append([sid, sx, x, round(t, 2), arah])
    pd.DataFrame(baris, columns=["shot_id", "shot_x_m", "receiver_x_m", "t_fb_ms", "direction"]).to_csv(INPUT_FILE, index=False)
    print("Picks SINTETIS dibuat di:", INPUT_FILE.resolve())

# ---------------- baca & validasi ----------------
SHA_BEFORE = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()
df = pd.read_csv(INPUT_FILE)
required = ["shot_id", "shot_x_m", "receiver_x_m", "t_fb_ms", "direction"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Kolom refraksi tidak ditemukan: {missing}")
df["direction"] = df["direction"].astype(str).str.lower().str.strip()
if not set(df["direction"].unique()).issubset({"forward", "reverse"}):
    raise ValueError("Kolom direction hanya boleh berisi forward/reverse.")
if df[required].isna().any().any():
    raise ValueError("Ada nilai kosong pada kolom wajib; periksa input tanpa menghapus baris.")

df["offset_m"] = (df["receiver_x_m"] - df["shot_x_m"]).abs()
n_nol = int((df["offset_m"] == 0).sum())
df_fit = df[df["offset_m"] > 0].copy()

rows_vel = []
rows_depth = []
catatan_picks = []
hasil = {}

for arah in ["forward", "reverse"]:
    d = df_fit[df_fit["direction"] == arah].sort_values("offset_m")
    x = d["offset_m"].to_numpy(dtype=float)
    t = d["t_fb_ms"].to_numpy(dtype=float)
    n = len(x)
    if n < 2 * MIN_PTS_SEG:
        raise ValueError(f"Picks arah {arah} terlalu sedikit ({n}).")

    # ---- breakpoint ----
    if BREAKPOINT_MODE == "manual":
        bp = BREAKPOINT_MANUAL_M[arah]
        k = int(np.sum(x <= bp))
        kriteria = f"manual: offset {bp:.2f} m"
    else:
        best_sse, k = np.inf, None
        for kk in range(MIN_PTS_SEG, n - MIN_PTS_SEG + 1):
            b1, a1 = np.polyfit(x[:kk], t[:kk], 1)
            b2, a2 = np.polyfit(x[kk:], t[kk:], 1)
            if b2 >= b1:          # slope refraksi harus lebih landai (V2 > V1)
                continue
            sse = np.sum((t[:kk] - (a1 + b1 * x[:kk]))**2) + np.sum((t[kk:] - (a2 + b2 * x[kk:]))**2)
            if sse < best_sse:
                best_sse, k = sse, kk
        if k is None:
            raise ValueError(f"Tidak ditemukan breakpoint dengan V2 > V1 pada arah {arah}.")
        kriteria = "otomatis: minimum total SSE dua garis, slope refraksi < slope langsung"
    if k < MIN_PTS_SEG or n - k < MIN_PTS_SEG:
        raise ValueError(f"Segmen terlalu pendek pada arah {arah} (k={k}). Ubah breakpoint.")

    # ---- regresi terpisah: t(ms) = a + b*x ----
    b_d, a_d = np.polyfit(x[:k], t[:k], 1)
    b_r, a_r = np.polyfit(x[k:], t[k:], 1)
    r2_d = 1 - np.sum((t[:k] - (a_d + b_d * x[:k]))**2) / np.sum((t[:k] - t[:k].mean())**2)
    r2_r = 1 - np.sum((t[k:] - (a_r + b_r * x[k:]))**2) / np.sum((t[k:] - t[k:].mean())**2)
    if b_d <= 0 or b_r <= 0:
        raise ValueError(f"Kemiringan regresi tidak positif pada {arah}; periksa picks/breakpoint.")

    V1 = 1000.0 / b_d
    V2 = 1000.0 / b_r
    if V2 <= V1:
        raise ValueError(f"V2 <= V1 pada {arah}; rumus intercept-time tidak valid.")

    ti = a_r                                     # intercept time (ms)
    xc = (a_r - a_d) / (b_d - b_r)               # crossover distance (m)
    h1 = (ti * 1e-3) * V1 * V2 / (2.0 * np.sqrt(V2**2 - V1**2))

    rows_vel.append({"direction": arah, "V1_m_s": V1, "V2_m_s": V2, "intercept_ms": ti,
                     "crossover_m": xc, "r2_direct": r2_d, "r2_refracted": r2_r,
                     "n_direct": k, "n_refracted": n - k})
    hasil[arah] = {"V1": V1, "V2": V2, "h1": h1, "xc": xc, "bp": 0.5 * (x[k - 1] + x[k]), "kriteria": kriteria}
    catatan_picks.append(f"{arah}: seluruh {n} picks dipakai (langsung={k}, bias={n - k}); tidak ada yang dikeluarkan.")

if n_nol > 0:
    catatan_picks.append(f"{n_nol} pick dengan offset 0 m dikecualikan dari regresi (tetap ada di file input).")

# ---- flag forward vs reverse ----
V2f, V2r = hasil["forward"]["V2"], hasil["reverse"]["V2"]
beda_v2 = abs(V2f - V2r) / (0.5 * (V2f + V2r)) * 100.0
flag_dir = "indikasi bidang miring atau heterogen" if beda_v2 > AMBANG_V2_PCT else "OK"
for arah in ["forward", "reverse"]:
    rows_depth.append({"direction": arah, "h1_m": hasil[arah]["h1"],
                       "V_ratio_V2_V1": hasil[arah]["V2"] / hasil[arah]["V1"], "flag": flag_dir})

pd.DataFrame(rows_vel).to_csv(OUT_DIR / "velocity_fit.csv", index=False)
pd.DataFrame(rows_depth).to_csv(OUT_DIR / "depth_estimate.csv", index=False)

# ---------------- verifikasi ----------------
V1s, V2s, h1s = 400.0, 1800.0, 5.0
xs = np.arange(2.0, 61.0, 2.0)
ti_v = 2 * h1s * np.sqrt(V2s**2 - V1s**2) / (V1s * V2s) * 1000.0
t_v = np.minimum(xs / V1s * 1000.0, xs / V2s * 1000.0 + ti_v)
k_v = int(np.sum(xs <= 2 * h1s * np.sqrt((V2s + V1s) / (V2s - V1s))))
b1, a1 = np.polyfit(xs[:k_v], t_v[:k_v], 1)
b2, a2 = np.polyfit(xs[k_v:], t_v[k_v:], 1)
V1e, V2e = 1000 / b1, 1000 / b2
h1e = a2 * 1e-3 * V1e * V2e / (2 * np.sqrt(V2e**2 - V1e**2))
err_s = np.array([abs(V1e / V1s - 1), abs(V2e / V2s - 1), abs(h1e / h1s - 1)]) * 100
cek_sintetik = bool(np.all(err_s <= TOL_SINTETIK_PCT))

cek_xc = []
for arah in ["forward", "reverse"]:
    h = hasil[arah]
    xc_pend = 2 * h["h1"] * np.sqrt((h["V2"] + h["V1"]) / (h["V2"] - h["V1"]))
    cek_xc.append(f"- Crossover {arah}: terhitung {h['xc']:.2f} m vs pendekatan {xc_pend:.2f} m (selisih {abs(h['xc'] - xc_pend) / xc_pend * 100:.1f} %).")

SHA_AFTER = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()

# ---------------- laporan & config ----------------
dh = abs(hasil["forward"]["h1"] - hasil["reverse"]["h1"])
baris_hasil = "\n".join(
    f"- {a}: V1 = {hasil[a]['V1']:.1f} m/s, V2 = {hasil[a]['V2']:.1f} m/s, crossover = {hasil[a]['xc']:.2f} m, h1 = {hasil[a]['h1']:.2f} m."
    for a in ["forward", "reverse"])
baris_bp = "\n".join(f"- {a}: breakpoint sekitar {hasil[a]['bp']:.2f} m ({hasil[a]['kriteria']})." for a in ["forward", "reverse"])
report = f"""# Laporan Seismik Refraksi (intercept-time, 2 lapisan, bidang bias datar)

## Pemilihan breakpoint
{baris_bp}

## Hasil
{baris_hasil}
- Beda V2 forward-reverse = {beda_v2:.1f} % (ambang {AMBANG_V2_PCT} %) -> {flag_dir}.
- Beda h1 forward-reverse = {dh:.2f} m (tidak dirata-ratakan).

## Picks
{chr(10).join('- ' + c for c in catatan_picks)}

## Keterbatasan
- Asumsi bidang bias datar, 2 lapisan, kecepatan konstan per lapisan; tanpa tomografi/inversi.
- Tidak digunakan untuk menetapkan kapasitas dukung tanah atau desain pondasi.

## Verifikasi
- Sintetis (galat V1, V2, h1 dalam %) = {np.round(err_s, 2)} -> {'LULUS' if cek_sintetik else 'GAGAL'}
{chr(10).join(cek_xc)}
- SHA-256 input sebelum/sesudah sama: {SHA_BEFORE == SHA_AFTER}
"""
(OUT_DIR / "report.md").write_text(report, encoding="utf-8")

config = {"input": str(INPUT_FILE), "model": "2 lapisan, bidang bias datar",
          "rumus_h1": "h1 = ti*V1*V2 / (2*sqrt(V2^2 - V1^2))", "rumus_crossover": "xc = (a_r - a_d)/(b_d - b_r)",
          "breakpoint_mode": BREAKPOINT_MODE, "breakpoint_hasil_m": {a: hasil[a]["bp"] for a in hasil},
          "min_pts_segmen": MIN_PTS_SEG, "ambang_beda_V2_pct": AMBANG_V2_PCT,
          "python_version": sys.version, "numpy_version": np.__version__, "pandas_version": pd.__version__,
          "sha256_before": SHA_BEFORE, "sha256_after": SHA_AFTER, "input_unchanged": SHA_BEFORE == SHA_AFTER}
(OUT_DIR / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print(f"Refraksi selesai. h1 forward = {hasil['forward']['h1']:.2f} m, reverse = {hasil['reverse']['h1']:.2f} m | "
      f"sintetis: {cek_sintetik} | hash sama: {SHA_BEFORE == SHA_AFTER}")
print(baris_hasil)
print("Hasil di:", OUT_DIR.resolve())

Folder kerja (BASE_DIR): D:\Semester 5\Metode AI
Refraksi selesai. h1 forward = 4.98 m, reverse = 5.05 m | sintetis: True | hash sama: True
- forward: V1 = 400.4 m/s, V2 = 1794.9 m/s, crossover = 12.48 m, h1 = 4.98 m.
- reverse: V1 = 399.1 m/s, V2 = 1819.1 m/s, crossover = 12.70 m, h1 = 5.05 m.
Hasil di: D:\Semester 5\Metode AI\results\refraction


## Cara Pakai
1. Pilih satu prompt dan isi semua `{{...}}` serta path file.
2. Jalankan, lalu nilai hasil aktual terhadap **[Q]** dan **[V]**.
3. Bila prompt terlalu panjang, pindahkan detail ke file configuration/recipe pendamping dan pertahankan **Constraints** serta **Quality criteria** di prompt.
4. Simpan diff dan `run_config.json` sebagai bukti perubahan.